# **01 - Synthetic Data Generation**

Created a synthetic dataset of 10,000 users with realistic behavioral distributions per segment, completed with initial EDA to validate data quality before entering the next pipeline.

## **1. Import Library**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os, warnings
from datetime import datetime, timedelta


✓ Semua folder siap
  Working directory: c:\Users\Lenovo\Documents\Defrizal\Project\Portofolio\Data\Data Analyst\Ride Hailing Loyalty Program — User Segmentation & Churn Risk Analysis\notebooks


## **2. Global Configurations**

In [ ]:
warnings.filterwarnings("ignore")
np.random.seed(42)

# Buat folder struktur proyek
for folder in ["../data/raw", "../data/processed", "../models", "../outputs/figures", "../outputs/reports"]:
    os.makedirs(folder, exist_ok=True)

print(f"  Working directory: {os.getcwd()}")

## **3. Definition of 5 Hidden Segments (Ground Truth)**

Each segment has a different parameter range to allow the model to learn real-world patterns. The churn distribution is based on loyalty program literature (Champions ~4%, Hibernating ~85%).

In [ ]:
SEGMENTS = {
    "Champions": {
        "n": 1500,
        "rec": (1, 14),           # recency_days
        "freq": (15, 30),         # frequency_monthly
        "spend": (800_000, 2_500_000),  # monetary_monthly (Rp)
        "churn_p": 0.04,
    },
    "Loyal": {
        "n": 2500,
        "rec": (7, 30),
        "freq": (8, 15),
        "spend": (300_000, 800_000),
        "churn_p": 0.10,
    },
    "At_Risk": {
        "n": 2000,
        "rec": (45, 90),
        "freq": (2, 6),
        "spend": (100_000, 350_000),
        "churn_p": 0.45,
    },
    "Promising": {
        "n": 2000,
        "rec": (3, 21),
        "freq": (3, 8),
        "spend": (80_000, 250_000),
        "churn_p": 0.25,
    },
    "Hibernating": {
        "n": 2000,
        "rec": (90, 365),
        "freq": (0, 1),
        "spend": (0, 80_000),
        "churn_p": 0.85,
    },
}

# Tampilkan ringkasan
print("Konfigurasi segmen:")
print(f"{'Segmen':<14} {'N':>5}  {'Recency':>12}  {'Freq/bln':>10}  {'Spend/bln':>20}  {'Churn%':>7}")
print("-" * 80)
for seg, p in SEGMENTS.items():
    print(f"{seg:<14} {p['n']:>5}  "
          f"{str(p['rec']):>12}  "
          f"{str(p['freq']):>10}  "
          f"Rp {p['spend'][0]:>8,} – {p['spend'][1]:>9,}  "
          f"{p['churn_p']*100:>6.0f}%")
print(f"{'TOTAL':<14} {sum(p['n'] for p in SEGMENTS.values()):>5}")


Konfigurasi segmen:
Segmen             N       Recency    Freq/bln             Spend/bln   Churn%
--------------------------------------------------------------------------------
Champions       1500       (1, 14)    (15, 30)  Rp  800,000 – 2,500,000       4%
Loyal           2500       (7, 30)     (8, 15)  Rp  300,000 –   800,000      10%
At_Risk         2000      (45, 90)      (2, 6)  Rp  100,000 –   350,000      45%
Promising       2000       (3, 21)      (3, 8)  Rp   80,000 –   250,000      25%
Hibernating     2000     (90, 365)      (0, 1)  Rp        0 –    80,000      85%
TOTAL          10000


## **4. Generate Dataset**

In [6]:
TODAY = datetime(2025, 1, 1)
SERVICES = ["GrabBike", "GrabCar", "GrabFood", "GrabMart", "GrabExpress"]
CITIES   = ["Jakarta", "Bandung", "Surabaya", "Medan", "Bekasi"]
CITY_W   = [0.45, 0.15, 0.15, 0.10, 0.15]

rows = []

for seg_name, p in SEGMENTS.items():
    for _ in range(p["n"]):
        recency   = np.random.randint(*p["rec"])
        last_tx   = TODAY - timedelta(days=recency)
        freq      = np.random.randint(*p["freq"]) if p["freq"][1] > 0 else 0
        monetary  = round(np.random.uniform(*p["spend"]), -3)
        ovo_pts   = int(monetary * 0.01 * np.random.uniform(0.8, 1.2))

        # Tier berdasarkan monetary
        if monetary > 1_500_000: tier = "Platinum"
        elif monetary > 600_000:  tier = "Gold"
        elif monetary > 200_000:  tier = "Silver"
        else:                     tier = "Member"

        # Services — multi-service lebih umum di Champions
        n_svc = np.random.choice([1,2,3,4], p=[0.1,0.3,0.4,0.2]) if seg_name=="Champions"            else np.random.choice([1,2,3,4], p=[0.4,0.3,0.2,0.1])
        services = np.random.choice(SERVICES, size=n_svc, replace=False).tolist()

        churn = int(np.random.random() < p["churn_p"])

        rows.append({
            "user_id"           : f"USR{len(rows):06d}",
            "segment_true"      : seg_name,
            "last_tx_date"      : last_tx.date(),
            "recency_days"      : recency,
            "frequency_monthly" : freq,
            "monetary_monthly"  : monetary,
            "ovo_points_balance": ovo_pts,
            "tier"              : tier,
            "services_used"     : "|".join(services),
            "n_services"        : len(services),
            "city"              : np.random.choice(CITIES, p=CITY_W),
            "churn_label"       : churn,
        })

df = pd.DataFrame(rows).sample(frac=1, random_state=42).reset_index(drop=True)
df.to_csv("../data/raw/rewards_synthetic.csv", index=False)

print(f"Dataset shape   : {df.shape}")
print(f"Churn rate      : {df.churn_label.mean():.1%}")
print(f"Kolom           : {list(df.columns)}")
print()
print(df["segment_true"].value_counts())


Dataset shape   : (10000, 12)
Churn rate      : 34.4%
Kolom           : ['user_id', 'segment_true', 'last_tx_date', 'recency_days', 'frequency_monthly', 'monetary_monthly', 'ovo_points_balance', 'tier', 'services_used', 'n_services', 'city', 'churn_label']

segment_true
Loyal          2500
Promising      2000
At_Risk        2000
Hibernating    2000
Champions      1500
Name: count, dtype: int64
